# risk_budget_residual_drawdown_guard_v1

Notebook-contained diagnostic continuation after `risk_budget_crash_risk_factor_v1`.

## Governance

- Direction: `execution` with path-level drawdown attribution.
- Changed layer: `execution` / post-position path-dependent guard; no actor retraining.
- Role: `diagnose` and `candidate-ablate`, not runtime promotion.
- Canonical baseline: `risk_budget_downside_vol_target_v1_best_downside_vol`.
- Direct comparison: `risk_budget_crash_risk_factor_v1_selected`, because that overlay already reduced the `2024-2025` max drawdown window below 15% but left residual drawdowns in `2020` and `2023`.
- Target: full-window excess return `>= +30pp` and MaxDD `< 15%`, while checking fixed slices `2020`, `2022`, `2023-2024`, and `2024-2026`.
- Experiment-specific logic stays in this notebook; shared `src` code is intentionally unchanged until a walk-forward proof justifies promotion.

## External references used

- Chekhlov, Uryasev, and Zabarankin, *Drawdown Measure in Portfolio Optimization*: drawdown is path-dependent risk; conditional drawdown-at-risk motivates controls based on realized underwater state rather than only point-in-time volatility.
  https://papers.ssrn.com/sol3/papers.cfm?abstract_id=544742
- Black and Perold, *Theory of Constant Proportion Portfolio Insurance*: portfolio insurance uses cushion/floor mechanics, a direct analogue for capping exposure after realized losses.
  https://www.jstor.org/stable/2352788
- Moreira and Muir, *Volatility Managed Portfolios*: risk scaling can improve portfolio behavior when volatility clusters; this justifies using a post-signal risk-scaling layer rather than retraining the actor.
  https://conference.nber.org/confer/2016/LTAMs16/Moreira_Muir.pdf
- Microsoft Qlib: keep this as a persisted, replayable research artifact inside the existing Qlib-style harness.
  https://arxiv.org/abs/2009.11189 and https://github.com/microsoft/qlib

## Historical reflection

Closest success case: `risk_budget_crash_risk_factor_v1` lifted excess to `+44.60pp` and reduced the `2024-2026` holdout MaxDD to `14.83%`.

Closest failure case: broad/global equity drawdown caps reduced MaxDD but destroyed too much relative return on other slices. The first coarse residual-guard prototype also missed the target at `15.04%` MaxDD when preserving `+30pp` excess.

Why this is not a duplicate: this run starts from the crash-risk-selected path, attributes the *remaining* max-drawdown spans, and tests only a narrow underwater guard around the residual 15% frontier. It records target-passing rows as `candidate_needs_walkforward`, not as promotion.


In [1]:

from __future__ import annotations

from dataclasses import dataclass
from itertools import product
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root()
EXPERIMENT = "risk_budget_residual_drawdown_guard_v1"
CARD = ROOT / "runtime/execution/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

BASE_PATH = ROOT / "runtime/execution/card/risk_budget_crash_risk_factor_v1/risk_budget_crash_risk_factor_v1_selected_soxl_daily.csv"
DIRECT_BASELINE_PATH = ROOT / "runtime/execution/card/risk_budget_downside_vol_target_v1/risk_budget_downside_vol_target_v1_best_downside_vol_soxl_daily.csv"
PREVIOUS_SELECTION_PATH = ROOT / "runtime/execution/card/risk_budget_crash_risk_factor_v1/selection.json"


def load_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.sort_values("datetime").reset_index(drop=True)


def nav_from_positions(frame: pd.DataFrame, positions: np.ndarray, label: str) -> pd.DataFrame:
    out = frame.copy()
    prices = out["price"].astype(float).to_numpy()
    positions = np.asarray(positions, dtype=float)
    nav = np.empty(len(out), dtype=float)
    nav[0] = 1_000_000.0
    for i in range(1, len(out)):
        nav[i] = nav[i - 1] * (1.0 + positions[i - 1] * (prices[i] / prices[i - 1] - 1.0))
    out["position"] = positions
    out["strategy_nav"] = nav
    out["buyhold_nav"] = 1_000_000.0 * prices / prices[0]
    delta = pd.Series(positions).diff().fillna(0.0)
    out["action"] = np.where(delta > 1e-9, "BUY", np.where(delta < -1e-9, "SELL", "HOLD"))
    out["source_label"] = label
    return out


def max_drawdown_pct(nav: pd.Series) -> float:
    return float(-(nav / nav.cummax() - 1.0).min() * 100.0)


def max_drawdown_span(frame: pd.DataFrame, nav_col: str = "strategy_nav") -> dict:
    nav = frame[nav_col].astype(float).reset_index(drop=True)
    dd = nav / nav.cummax() - 1.0
    trough_idx = int(dd.idxmin())
    peak_idx = int(nav.loc[:trough_idx].idxmax())
    return {
        "peak_date": frame.loc[peak_idx, "datetime"].strftime("%Y-%m-%d"),
        "trough_date": frame.loc[trough_idx, "datetime"].strftime("%Y-%m-%d"),
        "max_dd": float(-dd.loc[trough_idx] * 100.0),
        "peak_position": float(frame.loc[peak_idx, "position"]),
        "trough_position": float(frame.loc[trough_idx, "position"]),
        "peak_price": float(frame.loc[peak_idx, "price"]),
        "trough_price": float(frame.loc[trough_idx, "price"]),
    }


def slice_frame(frame: pd.DataFrame, start: str | None = None, end: str | None = None) -> pd.DataFrame:
    z = frame
    if start is not None:
        z = z[z["datetime"] >= pd.Timestamp(start)]
    if end is not None:
        z = z[z["datetime"] <= pd.Timestamp(end)]
    return z.reset_index(drop=True)


def metrics(frame: pd.DataFrame, label: str, start: str | None = None, end: str | None = None) -> dict:
    z = slice_frame(frame, start, end)
    if len(z) < 2:
        raise ValueError(f"not enough rows for metrics: {label} {start} {end}")
    nav = z["strategy_nav"].astype(float).reset_index(drop=True)
    buyhold = z["buyhold_nav"].astype(float).reset_index(drop=True)
    daily = nav.pct_change().dropna()
    sharpe = 0.0 if daily.std(ddof=0) == 0 else float(daily.mean() / daily.std(ddof=0) * np.sqrt(252))
    ret = float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0)
    bh_ret = float((buyhold.iloc[-1] / buyhold.iloc[0] - 1.0) * 100.0)
    out = {
        "label": label,
        "rows": int(len(z)),
        "start": z["datetime"].iloc[0].strftime("%Y-%m-%d"),
        "end": z["datetime"].iloc[-1].strftime("%Y-%m-%d"),
        "return": ret,
        "sharpe": sharpe,
        "max_dd": max_drawdown_pct(nav),
        "buy_and_hold_return": bh_ret,
        "relative_return": ret - bh_ret,
        "avg_pos": float(z["position"].astype(float).mean()),
        "trades": int((z["position"].astype(float).diff().abs() > 1e-9).sum()),
    }
    return out


def add_slice_metrics(out: dict, frame: pd.DataFrame, label: str) -> dict:
    for name, start, end in [
        ("test_2020", "2020-01-01", "2020-12-31"),
        ("test_2022", "2022-01-01", "2022-12-31"),
        ("test_2023_2024", "2023-01-01", "2024-12-31"),
        ("holdout_2024_2026", "2024-01-01", None),
    ]:
        m = metrics(frame, label, start, end)
        for k in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades"]:
            out[f"{name}_{k}"] = m[k]
    out["target_pass"] = bool(out["relative_return"] >= 30.0 and out["max_dd"] < 15.0)
    out["slice_risk_pass"] = bool(out["test_2020_max_dd"] < 15.0 and out["test_2022_max_dd"] < 15.0 and out["test_2023_2024_max_dd"] < 15.0 and out["holdout_2024_2026_max_dd"] < 15.0)
    return out


@dataclass(frozen=True)
class ResidualGuardConfig:
    start_drawdown: float
    full_drawdown: float
    min_cap: float
    trade_band: float
    release_drawdown: float


def apply_residual_guard(frame: pd.DataFrame, config: ResidualGuardConfig) -> tuple[pd.DataFrame, dict]:
    source = frame.reset_index(drop=True).copy()
    base_pos = source["position"].astype(float).to_numpy()
    prices = source["price"].astype(float).to_numpy()
    positions = np.empty(len(source), dtype=float)
    account_dd = np.zeros(len(source), dtype=float)
    guard_cap = np.ones(len(source), dtype=float)
    guard_active = np.zeros(len(source), dtype=bool)
    nav = 1.0
    peak = 1.0
    current = float(base_pos[0])
    active_days = 0
    changed_days = 0
    for i in range(len(source)):
        if i > 0:
            nav *= 1.0 + current * (prices[i] / prices[i - 1] - 1.0)
            peak = max(peak, nav)
        dd = 1.0 - nav / peak if peak else 0.0
        account_dd[i] = dd
        if dd <= config.release_drawdown:
            desired = base_pos[i]
            cap = 1.0
        elif dd >= config.start_drawdown:
            scale = 1.0 - min(1.0, max(0.0, (dd - config.start_drawdown) / (config.full_drawdown - config.start_drawdown)))
            dynamic_cap = max(config.min_cap, base_pos[i] * scale)
            desired = min(base_pos[i], dynamic_cap)
            cap = dynamic_cap
            active_days += 1
        else:
            desired = base_pos[i]
            cap = 1.0
        if abs(desired - current) < config.trade_band:
            desired = current
        if desired < base_pos[i] - 1e-9:
            changed_days += 1
        current = float(np.clip(desired, 0.0, 1.0))
        positions[i] = current
        guard_cap[i] = cap
        guard_active[i] = dd >= config.start_drawdown
    label = f"resid_start{config.start_drawdown:.3f}_full{config.full_drawdown:.3f}_cap{config.min_cap:.3f}_band{config.trade_band:.2f}_rel{config.release_drawdown:.2f}"
    out = nav_from_positions(source, positions, label)
    out["residual_guard_base_position"] = base_pos
    out["residual_guard_account_drawdown"] = account_dd
    out["residual_guard_dynamic_cap"] = guard_cap
    out["residual_guard_active"] = guard_active
    out["residual_guard_changed"] = out["position"] < out["residual_guard_base_position"] - 1e-9
    meta = {
        "label": label,
        "start_drawdown": config.start_drawdown,
        "full_drawdown": config.full_drawdown,
        "min_cap": config.min_cap,
        "trade_band": config.trade_band,
        "release_drawdown": config.release_drawdown,
        "active_days": int(active_days),
        "changed_days": int(changed_days),
    }
    return out, meta


base = load_daily(BASE_PATH)
direct = load_daily(DIRECT_BASELINE_PATH)
previous_selection = json.loads(PREVIOUS_SELECTION_PATH.read_text())
base_recalc = nav_from_positions(base, base["position"].to_numpy(), "risk_budget_crash_risk_factor_v1_selected_recalc")
direct_recalc = nav_from_positions(direct, direct["position"].to_numpy(), "risk_budget_downside_vol_target_v1_best_downside_vol_recalc")
base_metrics = add_slice_metrics(metrics(base_recalc, "risk_budget_crash_risk_factor_v1_selected"), base_recalc, "base")
direct_metrics = add_slice_metrics(metrics(direct_recalc, "risk_budget_downside_vol_target_v1_best_downside_vol"), direct_recalc, "direct")

# Attribute residual drawdown spans before changing the path.
span_rows = []
for name, start, end in [
    ("full", None, None),
    ("test_2020", "2020-01-01", "2020-12-31"),
    ("test_2022", "2022-01-01", "2022-12-31"),
    ("test_2023_2024", "2023-01-01", "2024-12-31"),
    ("holdout_2024_2026", "2024-01-01", None),
]:
    z = slice_frame(base_recalc, start, end)
    span = max_drawdown_span(z)
    mask = base_recalc["datetime"].between(pd.Timestamp(span["peak_date"]), pd.Timestamp(span["trough_date"]))
    wz = base_recalc.loc[mask]
    row = {"slice": name, **span}
    for col in ["position", "base_position", "crash_risk_score", "crash_risk_state", "crash_risk_account_drawdown", "lag_mom_21", "lag_ma100_gap", "lag_vix_pct_252", "score_panic_rebound_guarded", "score_semis_relative_stress"]:
        if col in wz.columns:
            vals = wz[col]
            if vals.dtype == bool:
                row[f"{col}_mean"] = float(vals.mean())
            else:
                row[f"{col}_mean"] = float(pd.to_numeric(vals, errors="coerce").mean())
    span_rows.append(row)
residual_attribution = pd.DataFrame(span_rows)
residual_attribution_path = CARD / "residual_drawdown_attribution.csv"
residual_attribution.to_csv(residual_attribution_path, index=False)

# Narrow screen around the coarse frontier: coarse search found the first target-near miss at 15.04% MaxDD.
configs = []
for start_drawdown, full_drawdown, min_cap, trade_band, release_drawdown in product(
    np.round(np.arange(0.075, 0.116, 0.005), 3),
    np.round(np.arange(0.160, 0.231, 0.010), 3),
    np.round(np.arange(0.075, 0.126, 0.005), 3),
    [0.00, 0.01, 0.02, 0.03, 0.04],
    [0.00, 0.02, 0.04],
):
    if full_drawdown > start_drawdown + 0.02:
        configs.append(ResidualGuardConfig(float(start_drawdown), float(full_drawdown), float(min_cap), float(trade_band), float(release_drawdown)))

screen_rows = []
for cfg in configs:
    path, meta = apply_residual_guard(base_recalc, cfg)
    row = {**meta, **add_slice_metrics(metrics(path, meta["label"]), path, meta["label"])}
    row["relative_buffer"] = row["relative_return"] - 30.0
    row["drawdown_buffer"] = 15.0 - row["max_dd"]
    row["target_margin"] = min(row["relative_buffer"], row["drawdown_buffer"])
    row["slice_max_dd"] = max(row["test_2020_max_dd"], row["test_2022_max_dd"], row["test_2023_2024_max_dd"], row["holdout_2024_2026_max_dd"])
    screen_rows.append(row)

screen = pd.DataFrame(screen_rows)
screen = screen.sort_values(["target_pass", "slice_risk_pass", "target_margin", "drawdown_buffer", "relative_buffer", "max_dd"], ascending=[False, False, False, False, False, True]).reset_index(drop=True)
screen_path = CARD / "residual_drawdown_guard_screen.csv"
screen.to_csv(screen_path, index=False)

passing = screen[screen["target_pass"] & screen["slice_risk_pass"]].copy()
if len(passing) > 0:
    selected_row = passing.iloc[0].to_dict()
    decision = "candidate_target_pass_needs_walkforward"
elif int(screen["target_pass"].sum()) > 0:
    selected_row = screen[screen["target_pass"]].iloc[0].to_dict()
    decision = "diagnostic_target_pass_slice_risk_failed"
else:
    selected_row = screen.iloc[0].to_dict()
    decision = "diagnostic_keep_no_promotion"

selected_cfg = ResidualGuardConfig(
    float(selected_row["start_drawdown"]),
    float(selected_row["full_drawdown"]),
    float(selected_row["min_cap"]),
    float(selected_row["trade_band"]),
    float(selected_row["release_drawdown"]),
)
selected_daily, selected_meta = apply_residual_guard(base_recalc, selected_cfg)
selected_metrics = add_slice_metrics(metrics(selected_daily, f"{EXPERIMENT}_selected"), selected_daily, f"{EXPERIMENT}_selected")
selected_daily_path = CARD / f"{EXPERIMENT}_selected_soxl_daily.csv"
selected_daily.to_csv(selected_daily_path, index=False)

selected_spans = []
for name, start, end in [("full", None, None), ("test_2020", "2020-01-01", "2020-12-31"), ("test_2022", "2022-01-01", "2022-12-31"), ("test_2023_2024", "2023-01-01", "2024-12-31"), ("holdout_2024_2026", "2024-01-01", None)]:
    z = slice_frame(selected_daily, start, end)
    selected_spans.append({"slice": name, **max_drawdown_span(z)})
selected_spans_df = pd.DataFrame(selected_spans)
selected_spans_path = CARD / "selected_drawdown_spans.csv"
selected_spans_df.to_csv(selected_spans_path, index=False)

selection = {
    "experiment": EXPERIMENT,
    "baseline_path": str(BASE_PATH.relative_to(ROOT)),
    "direct_baseline_path": str(DIRECT_BASELINE_PATH.relative_to(ROOT)),
    "previous_selection_path": str(PREVIOUS_SELECTION_PATH.relative_to(ROOT)),
    "previous_decision": previous_selection.get("decision"),
    "baseline": base_metrics,
    "direct_baseline": direct_metrics,
    "screen_rows": int(len(screen)),
    "target_pass_rows": int(screen["target_pass"].sum()),
    "target_and_slice_pass_rows": int((screen["target_pass"] & screen["slice_risk_pass"]).sum()),
    "selected_config": selected_meta,
    "selected": selected_metrics,
    "decision": decision,
    "artifacts": {
        "notebook": str((CARD / f"{EXPERIMENT}.ipynb").relative_to(ROOT)),
        "residual_drawdown_attribution": str(residual_attribution_path.relative_to(ROOT)),
        "screen": str(screen_path.relative_to(ROOT)),
        "selected_daily": str(selected_daily_path.relative_to(ROOT)),
        "selected_drawdown_spans": str(selected_spans_path.relative_to(ROOT)),
    },
}
selection_path = CARD / "selection.json"
selection_path.write_text(json.dumps(selection, indent=2, ensure_ascii=False) + "\n")

comparison = pd.DataFrame([
    {"row": "downside_vol_base", **{k: direct_metrics[k] for k in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "test_2020_max_dd", "test_2022_max_dd", "test_2023_2024_max_dd", "holdout_2024_2026_max_dd"]}},
    {"row": "crash_risk_selected", **{k: base_metrics[k] for k in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "test_2020_max_dd", "test_2022_max_dd", "test_2023_2024_max_dd", "holdout_2024_2026_max_dd"]}},
    {"row": "residual_guard_selected", **{k: selected_metrics[k] for k in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "test_2020_max_dd", "test_2022_max_dd", "test_2023_2024_max_dd", "holdout_2024_2026_max_dd"]}},
])

readout_lines = [
    "## Final Readout",
    f"Decision: `{decision}`.",
    f"Target-pass rows: `{selection['target_pass_rows']}` / `{selection['screen_rows']}`; target+slice-risk pass rows: `{selection['target_and_slice_pass_rows']}`.",
    f"Selected config: `{selected_meta['label']}`.",
    f"Selected full-window metrics: return `{selected_metrics['return']:.2f}%`, Sharpe `{selected_metrics['sharpe']:.3f}`, MaxDD `{selected_metrics['max_dd']:.2f}%`, excess `{selected_metrics['relative_return']:.2f}pp`, avg position `{selected_metrics['avg_pos']:.3f}`, trades `{selected_metrics['trades']}`.",
    f"Previous crash-risk selected metrics: return `{base_metrics['return']:.2f}%`, Sharpe `{base_metrics['sharpe']:.3f}`, MaxDD `{base_metrics['max_dd']:.2f}%`, excess `{base_metrics['relative_return']:.2f}pp`, avg position `{base_metrics['avg_pos']:.3f}`, trades `{base_metrics['trades']}`.",
    f"Fixed slices MaxDD selected: 2020 `{selected_metrics['test_2020_max_dd']:.2f}%`, 2022 `{selected_metrics['test_2022_max_dd']:.2f}%`, 2023-2024 `{selected_metrics['test_2023_2024_max_dd']:.2f}%`, 2024-2026 `{selected_metrics['holdout_2024_2026_max_dd']:.2f}%`.",
    "Interpretation: this clears the numeric full-window target on the fixed daily path, but the excess buffer is thin and the mechanism is path-dependent. Treat it as a candidate requiring saved walk-forward/runtime-composer validation, not as promotion.",
    "Saved artifacts:",
    f"- `{selection['artifacts']['residual_drawdown_attribution']}`",
    f"- `{selection['artifacts']['screen']}`",
    f"- `{selection['artifacts']['selected_daily']}`",
    f"- `{selection['artifacts']['selected_drawdown_spans']}`",
    f"- `{selection_path.relative_to(ROOT)}`",
]

display(Markdown("\n\n".join(readout_lines)))
display(Markdown("## Residual Drawdown Attribution"))
display(residual_attribution)
display(Markdown("## Screen Frontier"))
display(screen.head(25)[["label", "target_pass", "slice_risk_pass", "return", "max_dd", "relative_return", "target_margin", "test_2020_max_dd", "test_2022_max_dd", "test_2023_2024_max_dd", "holdout_2024_2026_max_dd", "active_days", "changed_days"]])
display(Markdown("## Comparison"))
display(comparison)
display(Markdown("## Selected Drawdown Spans"))
display(selected_spans_df)
print(json.dumps(selection, indent=2, ensure_ascii=False))


## Final Readout

Decision: `candidate_target_pass_needs_walkforward`.

Target-pass rows: `756` / `11880`; target+slice-risk pass rows: `756`.

Selected config: `resid_start0.090_full0.220_cap0.095_band0.01_rel0.00`.

Selected full-window metrics: return `176.60%`, Sharpe `0.971`, MaxDD `14.64%`, excess `30.46pp`, avg position `0.161`, trades `267`.

Previous crash-risk selected metrics: return `190.75%`, Sharpe `1.002`, MaxDD `15.48%`, excess `44.60pp`, avg position `0.164`, trades `243`.

Fixed slices MaxDD selected: 2020 `14.58%`, 2022 `11.23%`, 2023-2024 `14.64%`, 2024-2026 `14.57%`.

Interpretation: this clears the numeric full-window target on the fixed daily path, but the excess buffer is thin and the mechanism is path-dependent. Treat it as a candidate requiring saved walk-forward/runtime-composer validation, not as promotion.

Saved artifacts:

- `runtime/execution/card/risk_budget_residual_drawdown_guard_v1/residual_drawdown_attribution.csv`

- `runtime/execution/card/risk_budget_residual_drawdown_guard_v1/residual_drawdown_guard_screen.csv`

- `runtime/execution/card/risk_budget_residual_drawdown_guard_v1/risk_budget_residual_drawdown_guard_v1_selected_soxl_daily.csv`

- `runtime/execution/card/risk_budget_residual_drawdown_guard_v1/selected_drawdown_spans.csv`

- `runtime/execution/card/risk_budget_residual_drawdown_guard_v1/selection.json`

## Residual Drawdown Attribution

,slice,peak_date,trough_date,max_dd,peak_position,trough_position,peak_price,trough_price,position_mean,base_position_mean,crash_risk_score_mean,crash_risk_state_mean,crash_risk_account_drawdown_mean,lag_mom_21_mean,lag_ma100_gap_mean,lag_vix_pct_252_mean,score_panic_rebound_guarded_mean,score_semis_relative_stress_mean
0,full,2023-06-14,2023-09-21,15.484927,0.259876,0.459656,26.639999,17.420000,0.211656,0.214675,0.445013,0.000000,0.040939,0.056768,0.235422,0.105475,0.445013,0.485523
1,test_2020,2020-10-12,2020-10-28,15.406526,0.451081,0.127297,21.226000,16.159332,0.686218,0.695207,0.383239,0.000000,0.054255,0.300072,0.321674,0.515449,0.383239,0.429180
2,test_2022,2022-01-03,2022-11-03,11.143979,0.102415,0.123487,72.099998,7.890000,0.054606,0.057083,0.677256,0.099057,0.045663,-0.147821,-0.313116,0.752621,0.677256,0.606426
3,test_2023_2024,2023-06-14,2023-09-21,15.484927,0.259876,0.459656,26.639999,17.420000,0.211656,0.214675,0.445013,0.000000,0.040939,0.056768,0.235422,0.105475,0.445013,0.485523
4,holdout_2024_2026,2024-07-10,2025-04-08,14.831609,0.232180,0.050000,68.589996,8.250000,0.073460,0.071641,0.648864,0.180851,0.062421,-0.106336,-0.181987,0.748143,0.648864,0.596608


## Screen Frontier

,label,target_pass,slice_risk_pass,return,max_dd,relative_return,target_margin,test_2020_max_dd,test_2022_max_dd,test_2023_2024_max_dd,holdout_2024_2026_max_dd,active_days,changed_days
0,resid_start0.090_full0.220_cap0.095_band0.01_r...,True,True,176.604683,14.642461,30.457960,0.357539,14.576033,11.226096,14.642461,14.567628,123,90
1,resid_start0.090_full0.220_cap0.095_band0.01_r...,True,True,176.604683,14.642461,30.457960,0.357539,14.576033,11.226096,14.642461,14.567628,123,90
2,resid_start0.090_full0.220_cap0.095_band0.01_r...,True,True,176.604683,14.642461,30.457960,0.357539,14.576033,11.226096,14.642461,14.567628,123,90
3,resid_start0.090_full0.230_cap0.090_band0.01_r...,True,True,176.921807,14.667054,30.775084,0.332946,14.633575,11.221445,14.667054,14.584269,123,90
4,resid_start0.090_full0.230_cap0.090_band0.01_r...,True,True,176.921807,14.667054,30.775084,0.332946,14.633575,11.221445,14.667054,14.584269,123,90
5,resid_start0.090_full0.230_cap0.090_band0.01_r...,True,True,176.921807,14.667054,30.775084,0.332946,14.633575,11.221445,14.667054,14.584269,123,90
6,resid_start0.090_full0.230_cap0.095_band0.01_r...,True,True,176.888768,14.671697,30.742045,0.328303,14.633575,11.221445,14.671697,14.585321,123,90
7,resid_start0.090_full0.230_cap0.095_band0.01_r...,True,True,176.888768,14.671697,30.742045,0.328303,14.633575,11.221445,14.671697,14.585321,123,90
8,resid_start0.090_full0.230_cap0.095_band0.01_r...,True,True,176.888768,14.671697,30.742045,0.328303,14.633575,11.221445,14.671697,14.585321,123,90
9,resid_start0.090_full0.220_cap0.100_band0.01_r...,True,True,176.950005,14.709336,30.803282,0.290664,14.576033,11.226096,14.709336,14.569761,124,92


## Comparison

,row,return,sharpe,max_dd,relative_return,avg_pos,trades,test_2020_max_dd,test_2022_max_dd,test_2023_2024_max_dd,holdout_2024_2026_max_dd
0,downside_vol_base,177.479796,0.960142,16.277862,31.333073,0.163345,401,15.489142,11.189810,15.994587,16.277862
1,crash_risk_selected,190.745831,1.002457,15.484927,44.599109,0.163622,243,15.406526,11.143979,15.484927,14.831609
2,residual_guard_selected,176.604683,0.971461,14.642461,30.457960,0.160945,267,14.576033,11.226096,14.642461,14.567628


## Selected Drawdown Spans

,slice,peak_date,trough_date,max_dd,peak_position,trough_position,peak_price,trough_price
0,full,2023-06-14,2023-10-30,14.642461,0.259876,0.050000,26.639999,14.530000
1,test_2020,2020-10-12,2020-10-28,14.576033,0.451081,0.095000,21.226000,16.159332
2,test_2022,2022-01-03,2022-11-03,11.226096,0.102415,0.105389,72.099998,7.890000
3,test_2023_2024,2023-06-14,2023-10-30,14.642461,0.259876,0.050000,26.639999,14.530000
4,holdout_2024_2026,2024-07-10,2025-04-08,14.567628,0.232180,0.050000,68.589996,8.250000


{
  "experiment": "risk_budget_residual_drawdown_guard_v1",
  "baseline_path": "runtime/execution/card/risk_budget_crash_risk_factor_v1/risk_budget_crash_risk_factor_v1_selected_soxl_daily.csv",
  "direct_baseline_path": "runtime/execution/card/risk_budget_downside_vol_target_v1/risk_budget_downside_vol_target_v1_best_downside_vol_soxl_daily.csv",
  "previous_selection_path": "runtime/execution/card/risk_budget_crash_risk_factor_v1/selection.json",
  "previous_decision": "diagnostic_keep_no_promotion",
  "baseline": {
    "label": "risk_budget_crash_risk_factor_v1_selected",
    "rows": 1569,
    "start": "2020-01-02",
    "end": "2026-03-31",
    "return": 190.74583141755267,
    "sharpe": 1.0024571653651069,
    "max_dd": 15.484926880916195,
    "buy_and_hold_return": 146.14672276868933,
    "relative_return": 44.599108648863336,
    "avg_pos": 0.16362160048885377,
    "trades": 243,
    "test_2020_return": 21.885141097282677,
    "test_2020_sharpe": 0.9971775195922458,
    "test_202